# Medical AI Fine-Tuning — Phi-3.5-mini-instruct + QLoRA

**TechCorp Challenge — Hackathon IA Ynov 2026**

Ce notebook fine-tune `microsoft/Phi-3.5-mini-instruct` sur le dataset médical `ruslanmv/ai-medical-chatbot` en utilisant **QLoRA** (Quantized Low-Rank Adaptation) pour un entraînement efficace sur GPU.

---

## Architecture

```
Phi-3.5-mini-instruct (3.8B params)
        |
        v
Quantization 4-bit NF4 (BitsAndBytes)
        |
        v
LoRA Adapters (rank=16, alpha=32)
   -> q_proj, v_proj uniquement
        |
        v
SFTTrainer (3 epochs, lr=2e-4)
        |
        v
Medical Phi-3.5 (adapters LoRA sauvegardés)
```

## Configuration Recommandée

| Paramètre | Valeur |
|-----------|--------|
| Runtime | GPU T4 (Google Colab) |
| VRAM requise | ~14 GB |
| Temps estimé | ~45 min (10k samples, 3 epochs) |
| Dataset | ruslanmv/ai-medical-chatbot (250k conversations) |

> **Avertissement :** Ce modèle est à des fins de recherche uniquement. Ne jamais utiliser pour des diagnostics médicaux réels sans validation par des professionnels de santé qualifiés.

## Cellule 1 — Installation des Dépendances

In [ ]:
# Cellule 1 — Installation des dépendances
# Durée estimée : 2-3 minutes sur Colab

!pip install -q \
    transformers>=4.40.0 \
    peft>=0.10.0 \
    trl>=0.8.6 \
    bitsandbytes>=0.43.0 \
    datasets>=2.19.0 \
    accelerate>=0.29.0

# Vérification de l'environnement GPU
import torch
import transformers
import peft
import trl

print('=' * 55)
print('VERIFICATION ENVIRONNEMENT')
print('=' * 55)
print(f'PyTorch      : {torch.__version__}')
print(f'Transformers : {transformers.__version__}')
print(f'PEFT         : {peft.__version__}')
print(f'TRL          : {trl.__version__}')
print(f'CUDA dispo   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU          : {torch.cuda.get_device_name(0)}')
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM         : {vram_gb:.1f} GB')
    if vram_gb < 12:
        print('ATTENTION : VRAM < 12GB — reduire batch_size a 2')
    else:
        print('OK : VRAM suffisante pour batch_size=4')
else:
    print('ATTENTION : Pas de GPU detecte ! Allez dans Execution > Modifier le type de runtime > GPU T4')
print()

## Cellule 2 — Chargement du Dataset Médical

In [ ]:
# Cellule 2 — Chargement du dataset ruslanmv/ai-medical-chatbot
#
# Dataset : ruslanmv/ai-medical-chatbot
#   - ~250,000 conversations médecin-patient
#   - Colonnes : 'Patient' (question), 'Doctor' (réponse)
#   - Source : dialogues médicaux authentiques
#   - Licence : cc-by-4.0
#
# On limite à MAX_SAMPLES pour le hackathon (entraînement rapide)

from datasets import load_dataset

# Paramètre : nombre d'exemples utilisés pour l'entraînement
MAX_SAMPLES = 10000  # Augmenter à 50000+ pour une meilleure qualité
TEST_SIZE   = 0.1   # 10% pour la validation
SEED        = 42

print('Chargement du dataset depuis HuggingFace Hub...')
print(f'  Dataset    : ruslanmv/ai-medical-chatbot')
print(f'  Max samples: {MAX_SAMPLES}')
print()

raw_dataset = load_dataset('ruslanmv/ai-medical-chatbot', split='train')
print(f'Taille totale du dataset : {len(raw_dataset):,} conversations')
print(f'Colonnes disponibles     : {raw_dataset.column_names}')

# Limiter le nombre de samples
if MAX_SAMPLES < len(raw_dataset):
    raw_dataset = raw_dataset.select(range(MAX_SAMPLES))
    print(f'Apres selection          : {len(raw_dataset):,} samples')

# Split train/eval
split = raw_dataset.train_test_split(test_size=TEST_SIZE, seed=SEED)
train_raw = split['train']
eval_raw  = split['test']

print(f'Train : {len(train_raw):,} exemples')
print(f'Eval  : {len(eval_raw):,} exemples')

# Afficher un exemple
print('\nExemple du dataset :')
sample = train_raw[0]
print(f'Patient : {str(sample.get("Patient", sample.get("question", ""))[:300])}...')
print(f'Doctor  : {str(sample.get("Doctor",  sample.get("answer",   ""))[:300])}...')

## Cellule 3 — Formatage en Format Alpaca

In [ ]:
# Cellule 3 — Formatage en format Alpaca instruction-following
#
# Template Alpaca standard :
#   ### Instruction: <description de la tâche>
#   ### Input: <question du patient>
#   ### Response: <réponse du médecin>
#
# Ce format est largement utilisé pour le SFT et est compatible
# avec Phi-3.5, Llama, Mistral et la plupart des LLMs open-source.

ALPACA_TEMPLATE = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
You are a knowledgeable and compassionate medical assistant. Answer the following medical question accurately and safely. Always remind the user to consult a healthcare professional for personal medical advice.

### Input:
{input}

### Response:
{output}"""


def format_alpaca(example):
    """Convertit un exemple du dataset en format Alpaca."""
    # Détecter les colonnes (dataset peut varier)
    if 'Patient' in example and 'Doctor' in example:
        patient_text = str(example['Patient']).strip()
        doctor_text  = str(example['Doctor']).strip()
    elif 'question' in example and 'answer' in example:
        patient_text = str(example['question']).strip()
        doctor_text  = str(example['answer']).strip()
    else:
        keys = list(example.keys())
        patient_text = str(example.get(keys[0], '')).strip()
        doctor_text  = str(example.get(keys[1], '')).strip() if len(keys) > 1 else ''

    # Filtrer les exemples vides ou trop courts
    if len(patient_text) < 10 or len(doctor_text) < 10:
        return {'text': ''}

    text = ALPACA_TEMPLATE.format(input=patient_text, output=doctor_text)
    return {'text': text}


print('Formatage des donnees en format Alpaca...')

train_dataset = train_raw.map(
    format_alpaca,
    remove_columns=train_raw.column_names,
    desc='Formatage train'
)
eval_dataset = eval_raw.map(
    format_alpaca,
    remove_columns=eval_raw.column_names,
    desc='Formatage eval'
)

# Filtrer les exemples vides
train_dataset = train_dataset.filter(lambda x: len(x['text']) > 50)
eval_dataset  = eval_dataset.filter(lambda x: len(x['text']) > 50)

print(f'Train apres formatage : {len(train_dataset):,} exemples')
print(f'Eval apres formatage  : {len(eval_dataset):,} exemples')

# Afficher un exemple complet
print('\n=== EXEMPLE FORMATE (extrait) ===')
print(train_dataset[0]['text'][:700])
print('...')

# Statistiques de longueur
lengths = [len(x['text'].split()) for x in train_dataset.select(range(min(1000, len(train_dataset))))]
print(f'\nStatistiques de longueur (mots) sur 1000 premiers exemples :')
print(f'  Min    : {min(lengths)}')
print(f'  Max    : {max(lengths)}')
print(f'  Moyenne: {sum(lengths)//len(lengths)}')

## Cellule 4 — Configuration QLoRA 4-bit + LoRA

In [ ]:
# Cellule 4 — Configuration QLoRA 4-bit + LoRA
#
# QLoRA = Quantized LoRA (Dettmers et al., 2023)
#
# Principe :
#   1. Charger le modèle en 4-bit NF4 (NormalFloat4)
#      -> Réduit la VRAM de ~15GB (fp16) à ~4GB
#   2. Ajouter des adaptateurs LoRA de rang faible
#      -> Seuls 0.1% des paramètres sont entraînés
#   3. Double quantization : quantifier les constantes de quantization
#      -> Économie supplémentaire de ~0.37 bits/param
#
# Hyperparamètres LoRA :
#   r (rank)   = 16  : dimension des matrices LoRA (A ∈ R^{d×r}, B ∈ R^{r×d})
#   alpha      = 32  : facteur d'échelle (scaling = alpha/r = 2)
#   dropout    = 0.05: régularisation sur les couches LoRA
#   target     = q_proj, v_proj : couches d'attention Query et Value

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_NAME = 'microsoft/Phi-3.5-mini-instruct'

# --- Tokenizer ---
print(f'Chargement du tokenizer : {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    padding_side='right',
)
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
print(f'Vocab size  : {len(tokenizer):,}')
print(f'Pad token   : {tokenizer.pad_token!r}')

# --- BitsAndBytes 4-bit Config ---
print('\nConfiguration quantization 4-bit NF4...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',               # NormalFloat4 — optimal pour poids LLM
    bnb_4bit_compute_dtype=torch.float16,    # Calculs en FP16 (plus rapide)
    bnb_4bit_use_double_quant=True,          # Double quant -> ~0.37 bits/param economises
)

# --- Chargement modèle 4-bit ---
print(f'\nChargement modele en 4-bit (peut prendre 3-5 min)...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',                      # Distribue automatiquement sur GPU(s)
    torch_dtype=torch.float16,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)

# Préparer pour k-bit training
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,        # Economise ~50% VRAM en recalculant les activations
)

# --- Config LoRA ---
print('\nConfiguration LoRA...')
lora_config = LoraConfig(
    r=16,                                   # Rang : balance capacité/mémoire
    lora_alpha=32,                          # Scaling factor (alpha/r = 2 = standard)
    lora_dropout=0.05,                      # Dropout léger pour régularisation
    target_modules=['q_proj', 'v_proj'],    # Query et Value projections de l'attention
    bias='none',                            # Pas de biais supplémentaires
    task_type='CAUSAL_LM',                 # Génération causale (GPT-style)
)

model = get_peft_model(model, lora_config)

# Statistiques
all_params       = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
pct = 100 * trainable_params / all_params

print(f'\n=== STATISTIQUES DU MODELE ===')
print(f'Parametres totaux       : {all_params:>15,}')
print(f'Parametres entrainables : {trainable_params:>15,} ({pct:.2f}%)')
print(f'Parametres geles        : {all_params-trainable_params:>15,} ({100-pct:.2f}%)')
print(f'LoRA rank               : {lora_config.r}')
print(f'LoRA alpha              : {lora_config.lora_alpha}')
print(f'Modules cibles          : {lora_config.target_modules}')
print(f'\nMemoire GPU utilisee    : {torch.cuda.memory_allocated()/1e9:.2f} GB')

## Cellule 5 — Configuration SFTTrainer

In [ ]:
# Cellule 5 — Configuration SFTTrainer (Supervised Fine-Tuning Trainer)
#
# SFTTrainer (TRL library) = wrapper spécialisé pour le SFT sur LLMs
#   - Gère automatiquement le champ 'text' du dataset
#   - Compatible PEFT/LoRA nativement
#   - Intègre gradient checkpointing et mixed precision
#
# Hyperparamètres clés :
#   batch_size = 4   | grad_accum = 4 -> effective batch = 16
#   lr = 2e-4        | schedule cosinus avec 5% warmup
#   epochs = 3       | early stopping implicite via save_best

import os
from trl import SFTConfig

OUTPUT_DIR    = './medical_phi35_lora'   # Répertoire de sauvegarde
NUM_EPOCHS    = 3                        # Epochs (3 = bon compromis qualité/temps)
BATCH_SIZE    = 4                        # Par GPU (T4 16GB : max 4 avec seq_len=512)
GRAD_ACCUM    = 4                        # Accumulation -> batch effectif = 16
LEARNING_RATE = 2e-4                     # Standard pour QLoRA fine-tuning
MAX_SEQ_LEN   = 512                      # Longueur max en tokens

os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Configuration SFTTrainer...')
print(f'  Output dir           : {OUTPUT_DIR}')
print(f'  Epochs               : {NUM_EPOCHS}')
print(f'  Batch size           : {BATCH_SIZE} x {GRAD_ACCUM} accum = {BATCH_SIZE*GRAD_ACCUM} effectif')
print(f'  Learning rate        : {LEARNING_RATE}')
print(f'  Max seq length       : {MAX_SEQ_LEN} tokens')

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,

    # Batch et accumulation de gradient
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,

    # Learning rate et scheduler
    learning_rate=LEARNING_RATE,
    lr_scheduler_type='cosine',          # Décroissance cosinus
    warmup_ratio=0.05,                   # 5% des steps = warmup linéaire

    # Optimisation mémoire
    gradient_checkpointing=True,         # Recalcule les activations -> -50% VRAM
    optim='paged_adamw_32bit',          # AdamW paginé (BitsAndBytes) -> moins de VRAM
    fp16=True,                           # Mixed precision FP16

    # Logging
    logging_steps=25,                    # Logger toutes les 25 steps
    logging_dir=os.path.join(OUTPUT_DIR, 'logs'),

    # Sauvegarde et évaluation
    save_strategy='epoch',               # Sauvegarder à chaque epoch
    evaluation_strategy='epoch',         # Évaluer à chaque epoch
    save_total_limit=2,                  # Garder les 2 meilleurs checkpoints
    load_best_model_at_end=True,         # Charger le meilleur modèle à la fin
    metric_for_best_model='eval_loss',
    greater_is_better=False,

    # Données
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field='text',           # Colonne contenant les prompts Alpaca
    packing=False,                       # Pas de packing (longueurs variables)

    # Divers
    remove_unused_columns=False,
    report_to='none',                    # Désactiver wandb (pas de clé API nécessaire)
    seed=42,
    dataloader_num_workers=2,
)

# Estimation du temps d'entraînement
steps_per_epoch = len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)
total_steps = steps_per_epoch * NUM_EPOCHS
est_time_min = total_steps * 3.5 / 60  # ~3.5 sec/step sur T4

print(f'\n=== ESTIMATION ENTRAINEMENT ===')
print(f'Steps par epoch      : {steps_per_epoch}')
print(f'Steps totaux         : {total_steps}')
print(f'Temps estime (T4)    : ~{est_time_min:.0f} minutes')
print('\nConfiguration SFTTrainer prete !')

## Cellule 6 — Lancement Entraînement + Affichage Métriques

In [ ]:
# Cellule 6 — Lancement de l'entraînement et affichage des métriques
#
# Le SFTTrainer va :
#   1. Tokenizer les exemples formatés
#   2. Lancer l'entraînement avec gradient checkpointing
#   3. Logger les métriques toutes les 25 steps
#   4. Sauvegarder le meilleur checkpoint à chaque epoch
#
# Métriques à surveiller :
#   train/loss : doit décroître régulièrement (< 1.5 est bon)
#   eval/loss  : doit suivre train/loss (si diverge = overfitting)
#   train/lr   : vérifie le schedule cosinus

from trl import SFTTrainer
from datetime import datetime
import matplotlib
matplotlib.use('Agg')  # Pour Colab
import matplotlib.pyplot as plt

print('Initialisation du SFTTrainer...')
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

print(f'\n{'='*55}')
print('LANCEMENT DE L ENTRAINEMENT')
print(f'{'='*55}')
print(f'Debut : {datetime.now().strftime("%H:%M:%S")}')
print('(Les logs apparaissent toutes les 25 steps)\n')

start_time = datetime.now()
train_result = trainer.train()
end_time = datetime.now()

duration = end_time - start_time
print(f'\nFin : {end_time.strftime("%H:%M:%S")}')
print(f'Duree totale : {duration}')

# Afficher les métriques finales
print(f'\n=== METRIQUES D ENTRAINEMENT ===')
for key, value in train_result.metrics.items():
    if isinstance(value, float):
        print(f'  {key:<35} : {value:.4f}')
    else:
        print(f'  {key:<35} : {value}')

# Évaluation finale
print(f'\n=== EVALUATION FINALE ===')
eval_results = trainer.evaluate()
for key, value in eval_results.items():
    if isinstance(value, float):
        print(f'  {key:<35} : {value:.4f}')
    else:
        print(f'  {key:<35} : {value}')

# Courbes de perte (si historique disponible)
try:
    history = trainer.state.log_history
    train_losses = [(h['step'], h['loss']) for h in history if 'loss' in h and 'eval_loss' not in h]
    eval_losses  = [(h['step'], h['eval_loss']) for h in history if 'eval_loss' in h]

    if train_losses:
        fig, ax = plt.subplots(1, 1, figsize=(10, 5))
        ax.plot([s for s,l in train_losses], [l for s,l in train_losses],
                label='Train Loss', color='blue', linewidth=2)
        if eval_losses:
            ax.plot([s for s,l in eval_losses], [l for s,l in eval_losses],
                    label='Eval Loss', color='orange', linewidth=2, marker='o')
        ax.set_xlabel('Steps')
        ax.set_ylabel('Loss')
        ax.set_title('Courbes de Perte — Medical Phi-3.5 Fine-Tuning')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=150)
        plt.show()
        print(f'\nCourbes sauvegardees : {OUTPUT_DIR}/training_curves.png')
except Exception as e:
    print(f'(Visualisation non disponible : {e})')

print('\nEntrainement termine !')

## Cellule 7 — Sauvegarde des Adapters LoRA

In [ ]:
# Cellule 7 — Sauvegarde des adaptateurs LoRA entraînés
#
# On sauvegarde UNIQUEMENT les adaptateurs LoRA (petits fichiers ~30-50MB)
# et non le modèle de base quantifié (trop volumineux pour Google Drive).
#
# Pour utiliser le modèle plus tard :
#   from peft import PeftModel
#   base = AutoModelForCausalLM.from_pretrained('microsoft/Phi-3.5-mini-instruct')
#   model = PeftModel.from_pretrained(base, './medical_phi35_lora')
#
# Pour merger les adapters dans le modèle de base (1 seul fichier) :
#   merged = model.merge_and_unload()
#   merged.save_pretrained('./medical_phi35_merged')

import os

SAVE_DIR = OUTPUT_DIR  # './medical_phi35_lora'

print(f'Sauvegarde des adapters LoRA dans : {SAVE_DIR}')

# Sauvegarder le modèle PEFT (adapters uniquement)
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

# Inventaire des fichiers sauvegardés
print('\nFichiers sauvegardes :')
total_size_mb = 0
for root, dirs, files in os.walk(SAVE_DIR):
    # Ignorer les sous-dossiers de checkpoints
    dirs[:] = [d for d in dirs if not d.startswith('checkpoint-')]
    for fname in sorted(files):
        fpath = os.path.join(root, fname)
        size_mb = os.path.getsize(fpath) / 1e6
        total_size_mb += size_mb
        rel_path = os.path.relpath(fpath, SAVE_DIR)
        print(f'  {rel_path:<45} {size_mb:>8.2f} MB')

print(f'\nTaille totale des adapters : {total_size_mb:.1f} MB')

# Optionnel : sauvegarder sur Google Drive (Colab)
print('\n--- Optionnel : Sauvegarde sur Google Drive ---')
print('Demonter le code ci-dessous pour activer :')
print('''
from google.colab import drive
drive.mount('/content/drive')
import shutil
drive_path = '/content/drive/MyDrive/medical_phi35_lora'
shutil.copytree(SAVE_DIR, drive_path, dirs_exist_ok=True)
print(f'Sauvegardes sur Drive : {drive_path}')
''')

print(f'\nAdapters LoRA sauvegardes avec succes dans : {SAVE_DIR}')
print(f'Pour recharger le modele fine-tune :')
print(f'  from peft import PeftModel')
print(f'  model = PeftModel.from_pretrained(base_model, "{SAVE_DIR}")')

## Cellule 8 — Test du Modèle Fine-Tuné

In [ ]:
# Cellule 8 — Test du modèle médical fine-tuné
#
# On teste le modèle avec 3 questions médicales couvrant :
#   1. Diagnostic différentiel (symptômes multiples)
#   2. Interactions médicamenteuses (pharmacologie)
#   3. Prévention (style de vie, diabète)
#
# Paramètres de génération :
#   temperature = 0.3 : Bas pour des réponses médicales précises (moins créatif)
#   top_p = 0.9       : Nucleus sampling (bon équilibre diversité/qualité)
#   max_new_tokens = 300 : Assez pour une réponse médicale complète

import torch

model.eval()  # Mode évaluation (désactive dropout)

test_questions = [
    {
        'id': 1,
        'domain': 'Diagnostic différentiel',
        'question': 'I have been experiencing persistent headaches for the past two weeks, mostly in the morning, along with some visual disturbances and neck stiffness. What could be causing these symptoms and should I see a doctor urgently?'
    },
    {
        'id': 2,
        'domain': 'Interactions médicamenteuses',
        'question': 'My doctor prescribed me ibuprofen 400mg for knee pain, but I am also taking warfarin 5mg daily for atrial fibrillation. Are there any risks in taking both medications together and what should I do?'
    },
    {
        'id': 3,
        'domain': 'Prévention diabète',
        'question': 'I am 45 years old, slightly overweight (BMI 27), with a family history of type 2 diabetes. My recent fasting blood sugar was 105 mg/dL. What lifestyle changes can I make to prevent developing diabetes?'
    },
]

print('=' * 60)
print('TEST DU MODELE MEDICAL FINE-TUNE')
print('=' * 60)

results = []

for item in test_questions:
    print(f'\n--- Question {item["id"]} : {item["domain"]} ---')
    print(f'Patient : {item["question"][:150]}...' if len(item['question']) > 150 else f'Patient : {item["question"]}')
    print()

    # Formater en Alpaca (sans la réponse — le modèle génère)
    prompt = f"""Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
You are a knowledgeable and compassionate medical assistant. Answer the following medical question accurately and safely. Always remind the user to consult a healthcare professional for personal medical advice.

### Input:
{item['question']}

### Response:
"""

    # Tokeniser
    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=512,
    )

    # Déplacer sur GPU si disponible
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}

    # Générer la réponse
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.3,           # Bas = plus factuel (important en médical)
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,    # Éviter les répétitions
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Décoder uniquement les nouveaux tokens
    input_len = inputs['input_ids'].shape[1]
    new_tokens = outputs[0][input_len:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    print(f'Docteur : {response}')
    results.append({'question': item['question'], 'response': response, 'domain': item['domain']})

    print()

print('=' * 60)
print('EVALUATION QUALITATIVE')
print('=' * 60)
print('''
Critères d'évaluation pour les réponses médicales :

1. Pertinence médicale
   - La réponse adresse bien la question posée
   - Terminologie médicale appropriée
   
2. Sécurité patient
   - Recommandation de consulter un médecin incluse
   - Pas de diagnostic définitif posé
   - Signaux d'alarme mentionnés si pertinent

3. Précision factuelle
   - Informations médicales correctes
   - Pas d'hallucinations sur médicaments/dosages

4. Structure et lisibilité
   - Réponse bien organisée
   - Langage adapté au grand public
''')
print('\nTests termines ! Modele medical pret.')

## Cellule Bonus — Utilisation Ultérieure du Modèle

In [ ]:
# Cellule Bonus — Charger et utiliser le modèle fine-tuné dans une nouvelle session
#
# Ce code montre comment recharger les adapters LoRA sauvegardés
# dans une nouvelle session Colab ou en local.

print('=== RECHARGEMENT DU MODELE FINE-TUNE ===')
print('Code a executer dans une nouvelle session :\n')

reload_code = '''
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = "microsoft/Phi-3.5-mini-instruct"
ADAPTER_DIR = "./medical_phi35_lora"  # Ou chemin Google Drive

# Config 4-bit pour économiser la VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# Charger le modèle de base
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Charger les adapters LoRA
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, trust_remote_code=True)

model.eval()
print("Modele medical charge et pret !")

# Générer une réponse
def ask_medical_question(question, max_tokens=300):
    prompt = f"""Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
You are a knowledgeable and compassionate medical assistant.

### Input:
{question}

### Response:
"""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.3,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )
    
    response_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(response_tokens, skip_special_tokens=True).strip()

# Test
answer = ask_medical_question("What are the symptoms of appendicitis?")
print(f"Reponse : {answer}")
'''

print(reload_code)
print('\n=== FIN DU NOTEBOOK ===')
print('Fichiers generes :')
print(f'  {OUTPUT_DIR}/ — Adapters LoRA + Tokenizer')
print(f'  {OUTPUT_DIR}/training_curves.png — Courbes de perte')